In [ ]:
library(affy)
library(GEOquery)
library(tidyverse)

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Bioba

In [2]:
# defining datasets
training_set   <- "GSE42568"
validation_set <- c("GSE21653", "GSE20711", "GSE88770")

curr <- training_set

In [3]:
# reading in .cel files
raw_data <- ReadAffy(celfile.path = paste0("../data/", curr))

# RMA normalize data
normalized_data <- rma(raw.data)

# get expression data
normalized_expr <- as.data.frame(exprs(normalized_data))

# Clean column names - extract just GSM IDs
colnames(normalized_expr) <- gsub("_.*", "", colnames(normalized_expr))

head(normalized_expr)


ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'object' in selecting a method for function 'probeNames': object 'raw.data' not found


In [ ]:
# map probe IDs to gene symbols
gse <- getGEO(curr, GSEMatrix = TRUE)

Found 1 file(s)

GSE42568_series_matrix.txt.gz

Using locally cached version: C:\Users\ramya\AppData\Local\Temp\Rtmp0iGjRm/GSE42568_series_matrix.txt.gz

Using locally cached version of GPL570 found here:
C:\Users\ramya\AppData\Local\Temp\Rtmp0iGjRm/GPL570.soft.gz 



In [ ]:
# fetch feature data to get ID - gene symbol mapping
feature_data <- gse[[paste0(curr, "_series_matrix.txt.gz")]]@featureData@data

# subset
feature_data <- feature_data[c(1, 11)]

head(feature_data)

,ID,Gene Symbol
,<chr>,<chr>
1007_s_at,1007_s_at,DDR1 /// MIR4640
1053_at,1053_at,RFC2
117_at,117_at,HSPA6
121_at,121_at,PAX8
1255_g_at,1255_g_at,GUCA1A
1294_at,1294_at,MIR5193 /// UBA7


In [ ]:
normalized_expr <- normalized_expr |>
  rownames_to_column(var = "ID") |>
  inner_join(., feature_data, by = "ID")

head(normalized_expr)

ERROR: Error: object '.' not found


In [ ]:
# For multiple probes → one gene: take average
# For one probe → multiple genes: delete

normalized_expr <- normalized_expr |>
  # Remove rows where gene symbol is empty or maps to multiple genes
  filter(!grepl("///", `Gene Symbol`)) |>
  filter(`Gene Symbol` != "") |>

  # Group by gene symbol and average across probes
  group_by(`Gene Symbol`) |>
  summarise(across(where(is.numeric), mean)) |>
  ungroup()

dim(normalized_expr)
head(normalized_expr)

[1] 21655   122

Gene Symbol,GSM1045191,GSM1045192,GSM1045193,GSM1045194,GSM1045195,GSM1045196,GSM1045197,GSM1045198,GSM1045199,⋯,GSM1045302,GSM1045303,GSM1045304,GSM1045305,GSM1045306,GSM1045307,GSM1045308,GSM1045309,GSM1045310,GSM1045311
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,5.061765,5.384287,5.462018,5.104647,4.940101,5.087535,5.647719,5.502169,4.935021,⋯,5.612234,5.074677,6.872525,8.020731,5.132484,5.815884,5.310448,5.293781,5.683640,5.880834
A1BG-AS1,4.213439,4.497735,5.442854,4.142935,3.836375,3.840813,4.108047,4.117811,4.374573,⋯,5.274847,4.386748,5.158049,4.650182,4.528774,4.627988,4.577176,4.533182,4.283014,4.635586
A1CF,2.960794,3.844863,3.603847,3.710878,2.919968,3.246907,2.954523,3.181512,3.115969,⋯,3.695726,3.247118,3.037334,3.657515,3.350718,2.967194,2.943969,3.404410,3.238819,2.898562
A2M,6.814863,6.845371,4.532851,7.000975,7.117619,6.683618,6.839982,7.182724,6.365128,⋯,5.206523,6.804822,6.542975,5.768323,5.881436,6.835486,6.455288,6.328355,6.584059,6.576534
A2M-AS1,4.973031,4.252506,3.920574,4.064233,5.378826,5.553602,5.198546,5.391066,5.675315,⋯,3.974318,4.694706,4.407413,3.791085,3.544270,4.649494,3.700493,3.785728,3.637337,4.267453
A2ML1,3.133812,3.408455,3.779359,3.546180,3.138060,3.030034,3.338255,3.241485,3.111028,⋯,4.034024,3.141456,3.195094,3.162890,3.291446,2.894613,2.764031,3.472512,3.094196,3.243091


In [ ]:
# access the clinical metadata
clinical <- pData(gse[[1]])

dim(clinical)
colnames(clinical)
head(clinical)

[1] 121  50

[1] "title"                               "geo_accession"                      
 [3] "status"                              "submission_date"                    
 [5] "last_update_date"                    "type"                               
 [7] "channel_count"                       "source_name_ch1"                    
 [9] "organism_ch1"                        "characteristics_ch1"                
[11] "characteristics_ch1.1"               "characteristics_ch1.2"              
[13] "characteristics_ch1.3"               "characteristics_ch1.4"              
[15] "characteristics_ch1.5"               "characteristics_ch1.6"              
[17] "characteristics_ch1.7"               "characteristics_ch1.8"              
[19] "characteristics_ch1.9"               "treatment_protocol_ch1"             
[21] "molecule_ch1"                        "extract_protocol_ch1"               
[23] "label_ch1"                           "label_protocol_ch1"                 
[25] "taxid_ch1"                           "hyb_protocol"                       
[27] "scan_protocol"                       "description"                        
[29] "data_processing"                     "platform_id"                        
[31] "contact_name"                        "contact_email"                      
[33] "contact_department"                  "contact_institute"                  
[35] "contact_address"                     "contact_city"                       
[37] "contact_zip/postal_code"             "contact_country"                    
[39] "supplementary_file"                  "data_row_count"                     
[41] "age:ch1"                             "er_status:ch1"                      
[43] "grade:ch1"                           "lymph node status:ch1"              
[45] "overall survival event:ch1"          "overall survival time_days:ch1"     
[47] "relapse free survival event:ch1"     "relapse free survival time_days:ch1"
[49] "size:ch1"                            "tissue:ch1"

,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,characteristics_ch1,⋯,age:ch1,er_status:ch1,grade:ch1,lymph node status:ch1,overall survival event:ch1,overall survival time_days:ch1,relapse free survival event:ch1,relapse free survival time_days:ch1,size:ch1,tissue:ch1
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
GSM1045191,"Normal breast, N1_15_12_04",GSM1045191,Public on May 26 2013,Nov 27 2012,Jul 30 2013,RNA,1,"Breast tissue, normal",Homo sapiens,tissue: normal breast,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,normal breast
GSM1045192,"Normal breast, N4_14_12_04",GSM1045192,Public on May 26 2013,Nov 27 2012,Jul 30 2013,RNA,1,"Breast tissue, normal",Homo sapiens,tissue: normal breast,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,normal breast
GSM1045193,"Normal breast, N5_15_12_04",GSM1045193,Public on May 26 2013,Nov 27 2012,Jul 30 2013,RNA,1,"Breast tissue, normal",Homo sapiens,tissue: normal breast,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,normal breast
GSM1045194,"Normal breast, N6_14_12_04",GSM1045194,Public on May 26 2013,Nov 27 2012,Jul 30 2013,RNA,1,"Breast tissue, normal",Homo sapiens,tissue: normal breast,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,normal breast
GSM1045195,"Normal breast, P1_15_12_04",GSM1045195,Public on May 26 2013,Nov 27 2012,Jul 30 2013,RNA,1,"Breast tissue, normal",Homo sapiens,tissue: normal breast,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,normal breast
GSM1045196,"Normal breast, P10_26_1_05",GSM1045196,Public on May 26 2013,Nov 27 2012,Jul 30 2013,RNA,1,"Breast tissue, normal",Homo sapiens,tissue: normal breast,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,normal breast


In [ ]:
# Extract relevant columns
clinical <- clinical[, c(
  "geo_accession",
  "tissue:ch1",
  "age:ch1",
  "grade:ch1",
  "size:ch1",
  "er_status:ch1",
  "lymph node status:ch1",
  "relapse free survival event:ch1",
  "relapse free survival time_days:ch1",
  "overall survival event:ch1",
  "overall survival time_days:ch1"
)]

# Rename columns
colnames(clinical) <- c(
  "sample_id",
  "tissue_type",
  "patient_age",
  "tumor_grade",
  "tumor_size",
  "er_status",
  "lymph_node_status",
  "relapse_free_event",
  "relapse_free_days",
  "overall_survival_event",
  "overall_survival_time"
)

dim(clinical)
colnames(clinical)
head(clinical)

[1] 121   9

[1] "sample_id"          "tissue_type"        "patient_age"       
[4] "tumor_grade"        "tumor_size"         "er_status"         
[7] "lymph_node_status"  "relapse_free_event" "relapse_free_days"

,sample_id,tissue_type,patient_age,tumor_grade,tumor_size,er_status,lymph_node_status,relapse_free_event,relapse_free_days
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
GSM1045191,GSM1045191,normal breast,NA,NA,NA,NA,NA,NA,NA
GSM1045192,GSM1045192,normal breast,NA,NA,NA,NA,NA,NA,NA
GSM1045193,GSM1045193,normal breast,NA,NA,NA,NA,NA,NA,NA
GSM1045194,GSM1045194,normal breast,NA,NA,NA,NA,NA,NA,NA
GSM1045195,GSM1045195,normal breast,NA,NA,NA,NA,NA,NA,NA
GSM1045196,GSM1045196,normal breast,NA,NA,NA,NA,NA,NA,NA


DO NOT DROP NULL VALUES

The rows with null values correspond to normal breast cells that don't require data inputs as they are not tumorous in the first place.

In [ ]:
# create datasets directory if it doesn't exist
dir.create("../datasets", recursive = TRUE, showWarnings = FALSE)

# save datasets in datasets folder for persistent storage
saveRDS(normalized_expr, "../datasets/expression_matrix_train.rds")
saveRDS(clinical, "../datasets/clinical_metadata_train.rds")